In [ ]:
import trimesh

mesh = trimesh.load("DSV_obj\DSV_Schenker_expanded_V3 (1).obj", force='mesh')
print("Vertices:", mesh.vertices.shape)
print("Faces:", mesh.faces.shape)
mesh.show()


Vertices: (31558, 3)
Faces: (30810, 3)


In [ ]:
mesh.export("DSV_Schenker_expanded_V3.obj")


In [1]:
import open3d as o3d
mesh = o3d.io.read_triangle_mesh("OHLF_obj\OHLF_v2.8.3 (1).obj", enable_post_processing=True)
o3d.visualization.draw_geometries([mesh], mesh_show_back_face=True)


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [8]:
import cv2

# Callback function to display coordinates
def click_event(event, x, y, flags, param):
    if event == cv2.EVENT_LBUTTONDOWN:
        print(f"Clicked coordinates: X={x}, Y={y}")

        # Draw a small circle where you clicked
        cv2.circle(img, (x, y), 5, (0, 0, 255), -1)

        # Optionally display coordinates on image
        font = cv2.FONT_HERSHEY_SIMPLEX
        cv2.putText(img, f"({x},{y})", (x+10, y-10), font, 0.5, (255, 255, 255), 1)

        # Update image
        cv2.imshow("Image", img)

# Load your image
img = cv2.imread("undistorted_fisheye.png")

# Create window and set mouse callback
cv2.imshow("Image", img)
cv2.setMouseCallback("Image", click_event)

# Keep window open until key pressed
cv2.waitKey(0)
cv2.destroyAllWindows()


In [12]:
ohlf= cv2.imread("ohlf.png")
ohlf.shape

(1080, 1920, 3)

In [10]:
ohlf= cv2.imread("undistorted_fisheye.png")
ohlf.shape

(2592, 2592, 3)

In [37]:
import cv2
import numpy as np

# === Load Images ===
img_src = cv2.imread("undistorted_fisheye.png")          # source image (2592, 2592)
img_dst = cv2.imread("ohlf.png")     # destination image (1080, 1920, 3)


In [38]:
# === Resize large source image for easier clicking ===
scale = 0.5  # reduce to half (change to 0.3 or 0.4 if still too large)
display_src = cv2.resize(img_src, (0, 0), fx=scale, fy=scale)
display_src_vis = display_src 
img_dst_vis = img_dst


In [26]:
# === Lists to store clicked points ===
src_points_scaled = []   # points on scaled (displayed) image
dst_points = []          # points on destination image

# === Callback functions ===
def click_src(event, x, y, flags, param):
    if event == cv2.EVENT_LBUTTONDOWN:
        src_points_scaled.append((x, y))
        cv2.circle(display_src, (x, y), 5, (0, 0, 255), -1)
        cv2.imshow("Source (Scaled)", display_src)
        print(f"[SRC scaled] ({x}, {y})")

def click_dst(event, x, y, flags, param):
    if event == cv2.EVENT_LBUTTONDOWN:
        dst_points.append((x, y))
        cv2.circle(img_dst, (x, y), 5, (255, 0, 0), -1)
        cv2.imshow("Destination", img_dst)
        print(f"[DST] ({x}, {y})")


In [27]:
# === Select points in both images ===
cv2.imshow("Source (Scaled)", display_src)
cv2.setMouseCallback("Source (Scaled)", click_src)
cv2.waitKey(0)
cv2.destroyWindow("Source (Scaled)")

cv2.imshow("Destination", img_dst)
cv2.setMouseCallback("Destination", click_dst)
cv2.waitKey(0)
cv2.destroyAllWindows()


[SRC scaled] (859, 766)
[SRC scaled] (835, 751)
[SRC scaled] (912, 573)
[SRC scaled] (787, 502)
[SRC scaled] (681, 557)
[SRC scaled] (665, 578)
[SRC scaled] (623, 625)
[SRC scaled] (585, 696)
[SRC scaled] (555, 788)
[DST] (1304, 579)
[DST] (1307, 592)
[DST] (1210, 593)
[DST] (1210, 687)
[DST] (1258, 721)
[DST] (1268, 723)
[DST] (1297, 732)
[DST] (1327, 729)
[DST] (1366, 725)


In [53]:
# === Rescale source points back to original size ===
src_points = np.array([[x / scale, y / scale] for (x, y) in src_points_scaled], dtype=np.float32)
dst_points = np.array(dst_points, dtype=np.float32)

print("\n--- Points Collected ---")
print("Source (original):", src_points)
print("Destination:", dst_points)

# === Compute Homography ===
H, mask = cv2.findHomography(src_points, dst_points, cv2.RANSAC)
print("\nHomography Matrix:\n", H)



--- Points Collected ---
Source (original): [[1718. 1532.]
 [1670. 1502.]
 [1824. 1146.]
 [1574. 1004.]
 [1362. 1114.]
 [1330. 1156.]
 [1246. 1250.]
 [1170. 1392.]
 [1110. 1576.]]
Destination: [[1304.  579.]
 [1307.  592.]
 [1210.  593.]
 [1210.  687.]
 [1258.  721.]
 [1268.  723.]
 [1297.  732.]
 [1327.  729.]
 [1366.  725.]]

Homography Matrix:
 [[-3.76090664e-01 -7.63464200e-03  1.22937009e+03]
 [-2.88827250e-01 -1.18153422e-01  9.30985950e+02]
 [-2.48627269e-04 -8.78331920e-05  1.00000000e+00]]


In [29]:
# === Example: map a test point ===
test_point = np.array([[[1000, 1000]]], dtype=np.float32)
mapped = cv2.perspectiveTransform(test_point, H)
print("\nMapped test point (in destination image):", mapped[0][0])



Mapped test point (in destination image): [1274.4452  789.7122]


In [ ]:

warped = cv2.warpPerspective(img_src, H, (img_dst.shape[1], img_dst.shape[0]))
cv2.imshow("Warped Source → Destination View", warped) 
cv2.waitKey(0)
cv2.destroyAllWindows()


In [ ]:
import cv2
import numpy as np

# === Load Images ===
img_src = cv2.imread("undistorted_fisheye.png")  
img_dst = cv2.imread("ohlf.png")        

# === Resize source for usability ===
scale = 0.5
display_src = cv2.resize(img_src, (0, 0), fx=scale, fy=scale)

# === Compute or load Homography (already available in your case) ===
# H, mask = cv2.findHomography(src_points, dst_points, cv2.RANSAC)
# Example: replace this with your own matrix # Or directly paste your H here

# === Create copies for visualization ===
src_vis = display_src.copy()
dst_vis = img_dst.copy()

# === Define callback for clicks on source image ===
def click_source(event, x, y, flags, param):
    global src_vis, dst_vis

    if event == cv2.EVENT_LBUTTONDOWN:
        # Convert scaled point → original source coordinate
        x_orig, y_orig = x / scale, y / scale
        src_pt = np.array([[[x_orig, y_orig]]], dtype=np.float32)

        # Map via homography
        mapped_pt = cv2.perspectiveTransform(src_pt, H)
        mx, my = int(mapped_pt[0][0][0]), int(mapped_pt[0][0][1])

        # Draw green point on source
        cv2.circle(src_vis, (x, y), 8, (0, 255, 0), -1)
        cv2.putText(src_vis, f"({int(x_orig)}, {int(y_orig)})",
                    (x + 10, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)

        # Draw red point on destination
        cv2.circle(dst_vis, (mx, my), 8, (0, 0, 255), -1)
        cv2.putText(dst_vis, f"({mx}, {my})",
                    (mx + 10, my - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)

        # Update both windows
        cv2.imshow("Source Image (click here)", src_vis)
        cv2.imshow("Destination Image (mapped points)", dst_vis)

# === Create windows ===
cv2.namedWindow("Source Image (click here)", cv2.WINDOW_NORMAL)
cv2.namedWindow("Destination Image (mapped points)", cv2.WINDOW_NORMAL)

# === Show initial views ===
cv2.imshow("Source Image (click here)", src_vis)
cv2.imshow("Destination Image (mapped points)", dst_vis)

# === Attach callback to the SOURCE window ===
cv2.setMouseCallback("Source Image (click here)", click_source)

print("✅ Click on the SOURCE image — mapped points will appear in the DESTINATION image.")
print("Press ESC to exit.")

# === Main loop ===
while True:
    key = cv2.waitKey(1) & 0xFF
    if key == 27:  # ESC
        break

cv2.destroyAllWindows()


✅ Click on the SOURCE image — mapped points will appear in the DESTINATION image.
Press ESC to exit.


In [51]:
import cv2
import numpy as np

# === Load Images ===
img_src = cv2.imread("undistorted_fisheye.png")  # source (e.g. 2592x2592)
img_dst = cv2.imread("ohlf.png")                 # destination (1080x1920, 3)

# === Resize large source image for usability ===
scale = 0.5  # downscale factor for source display
display_src = cv2.resize(img_src, (0, 0), fx=scale, fy=scale)

# === Load or compute Homography ===
# Replace with your actual homography matrix
# H, mask = cv2.findHomography(src_points, dst_points, cv2.RANSAC)
H = H  # Example: load from file

# === Create copies for visualization ===
src_vis = display_src.copy()
dst_vis = img_dst.copy()

# === Combine into one display canvas ===
src_resized = cv2.resize(src_vis, (img_dst.shape[1], img_dst.shape[0]))
combined = cv2.hconcat([src_resized, dst_vis])

# Final display shrink factor to fit on screen
display_scale = 0.6  # adjust as needed for screen size
canvas_w = int(combined.shape[1] * display_scale)
canvas_h = int(combined.shape[0] * display_scale)
combined_display = cv2.resize(combined, (canvas_w, canvas_h))

# Store left image width for detecting click side
left_width = int(img_dst.shape[1] * display_scale)

# === Callback Function ===
def click_and_map(event, x, y, flags, param):
    global src_vis, dst_vis, combined_display

    if event == cv2.EVENT_LBUTTONDOWN:
        # Click within the LEFT (source) half
        if x < left_width:
            # Convert click position from display canvas → scaled source
            x_scaled = int(x / display_scale)
            y_scaled = int(y / display_scale)

            # Convert to original (unscaled) coordinates
            x_orig, y_orig = x_scaled / scale, y_scaled / scale
            src_pt = np.array([[[x_orig, y_orig]]], dtype=np.float32)

            # Apply homography
            mapped_pt = cv2.perspectiveTransform(src_pt, H)
            mx, my = int(mapped_pt[0][0][0]), int(mapped_pt[0][0][1])

            # Draw on source (green)
            cv2.circle(src_vis, (x_scaled, y_scaled), 8, (0, 255, 0), -1)
            cv2.putText(src_vis, f"({int(x_orig)}, {int(y_orig)})",
                        (x_scaled + 10, y_scaled - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)

            # Draw on destination (red)
            cv2.circle(dst_vis, (mx, my), 8, (0, 0, 255), -1)
            cv2.putText(dst_vis, f"({mx}, {my})",
                        (mx + 10, my - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)

            # Combine and resize for visualization
            src_resized = cv2.resize(src_vis, (img_dst.shape[1], img_dst.shape[0]))
            combined = cv2.hconcat([src_resized, dst_vis])
            combined_display = cv2.resize(combined, (canvas_w, canvas_h))

            cv2.imshow("Source (left) → Destination (right)", combined_display)

# === Show Initial Window ===
cv2.namedWindow("Source (left) → Destination (right)", cv2.WINDOW_NORMAL)
cv2.imshow("Source (left) → Destination (right)", combined_display)
cv2.setMouseCallback("Source (left) → Destination (right)", click_and_map)

print("✅ Click on the LEFT (source) image — mapped points will appear on the RIGHT (destination).")
print("Press ESC to exit.")

# === Main Loop ===
while True:
    key = cv2.waitKey(1) & 0xFF
    if key == 27:  # ESC key to exit
        break

cv2.destroyAllWindows()


✅ Click on the LEFT (source) image — mapped points will appear on the RIGHT (destination).
Press ESC to exit.


In [ ]:
import cv2
import os
video_path = r"C:\Users\Nahush\Desktop\Defect\part1\MicrosoftTeams-video.mp4"
output_folder = "frames"
os.makedirs(output_folder, exist_ok=True)
cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS)
print("Video FPS:", fps)

frame_number = 0
saved_frame_count = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break
    if frame_number % int(fps) == 0:
        filename = f"{output_folder}/frame_{saved_frame_count}.jpg"
        cv2.imwrite(filename, frame)
        print("Saved:", filename)
        saved_frame_count += 1

    frame_number += 1

cap.release()
print("Done!")



Video FPS: 30.0
Saved: frames/frame_0.jpg
Saved: frames/frame_1.jpg
Saved: frames/frame_2.jpg
Saved: frames/frame_3.jpg
Saved: frames/frame_4.jpg
Saved: frames/frame_5.jpg
Saved: frames/frame_6.jpg
Saved: frames/frame_7.jpg
Saved: frames/frame_8.jpg
Saved: frames/frame_9.jpg
Saved: frames/frame_10.jpg
Saved: frames/frame_11.jpg
Saved: frames/frame_12.jpg
Saved: frames/frame_13.jpg
Saved: frames/frame_14.jpg
Saved: frames/frame_15.jpg
Saved: frames/frame_16.jpg
Saved: frames/frame_17.jpg
Saved: frames/frame_18.jpg
Saved: frames/frame_19.jpg
Done!


In [1]:
from ids_peak import ids_peak
from ids_peak import ids_peak_ipl
import numpy as np
import cv2

ids_peak.Library.Initialize()

device_manager = ids_peak.DeviceManager.Instance()
device_manager.Update()
device = device_manager.Devices()[0].OpenDevice()

datastream = device.DataStreams()[0].OpenDataStream()
payload = device.Nodemap()["PayloadSize"].Value()
datastream.StartAcquisition()

buffer = datastream.WaitForFinishedBuffer(5000)
img = buffer.GetImage()
frame = img.get_numpy_array()

cv2.imshow("Frame", frame)
cv2.waitKey(1)

datastream.StopAcquisition()
device.Close()
ids_peak.Library.Close()


ModuleNotFoundError: No module named 'ids_peak'

In [12]:
import cv2
import os
video_path = r"C:\Users\Nahush\Videos\Part6_light.avi"
output_folder = "frames_training"
os.makedirs(output_folder, exist_ok=True)
cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS)
print("Video FPS:", fps)

frame_number = 0
saved_frame_count = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break
    if frame_number % int(10) == 0:
        filename = f"{output_folder}/frame_6_{saved_frame_count}.jpg"

        cv2.imwrite(filename, frame)
        print("Saved:", filename)
        saved_frame_count += 1

    frame_number += 1

cap.release()
print("Done!")

Video FPS: 41.0
Saved: frames_training/frame_6_0.jpg
Saved: frames_training/frame_6_1.jpg
Saved: frames_training/frame_6_2.jpg
Saved: frames_training/frame_6_3.jpg
Saved: frames_training/frame_6_4.jpg
Saved: frames_training/frame_6_5.jpg
Saved: frames_training/frame_6_6.jpg
Saved: frames_training/frame_6_7.jpg
Saved: frames_training/frame_6_8.jpg
Saved: frames_training/frame_6_9.jpg
Saved: frames_training/frame_6_10.jpg
Saved: frames_training/frame_6_11.jpg
Saved: frames_training/frame_6_12.jpg
Saved: frames_training/frame_6_13.jpg
Saved: frames_training/frame_6_14.jpg
Saved: frames_training/frame_6_15.jpg
Saved: frames_training/frame_6_16.jpg
Saved: frames_training/frame_6_17.jpg
Saved: frames_training/frame_6_18.jpg
Saved: frames_training/frame_6_19.jpg
Saved: frames_training/frame_6_20.jpg
Saved: frames_training/frame_6_21.jpg
Saved: frames_training/frame_6_22.jpg
Saved: frames_training/frame_6_23.jpg
Saved: frames_training/frame_6_24.jpg
Saved: frames_training/frame_6_25.jpg
Saved: